# Selección e integración de los datos

## Objetivo

Este notebook desarrolla la fase de **Selección de datos** de la metodología KDD. Su objetivo consiste en verificar la estructura de los conjuntos de datos inmobiliarios de las diez ciudades, reconocer las variables comunes y las específicas e integrar todos los archivos en un solo conjunto de datos.

La fase concluye con un DataFrame unificado que conserva la ciudad de procedencia de cada anuncio y que se utilizará en el notebook de preprocesamiento.


## 1. Importación de librerías

Se utilizan `Path` para gestionar rutas de forma portable y `pandas` para cargar, revisar e integrar los archivos Excel.

In [ ]:
# =====================================================
# IMPORTACIÓN DE LIBRERÍAS
# =====================================================

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


### Interpretación

Las librerías necesarias para esta fase se han cargado correctamente. `Path` se empleará para trabajar con rutas de manera reproducible, y `pandas` facilitará la lectura, la revisión y la integración de los archivos procedentes de las distintas ciudades.


## 2. Configuración del proyecto

Se definen las carpetas principales del proyecto. La estructura esperada es:

- `data/raw`: archivos originales de las ciudades;
- `data/interim`: datos intermedios, si fueran necesarios;
- `data/processed`: conjunto de datos final después del preprocesamiento.

Las rutas relativas facilitan que el proyecto pueda ejecutarse en distintos equipos sin modificar direcciones absolutas.

In [ ]:
# =====================================================
# DETECCIÓN DE LA RAÍZ DEL PROYECTO
# =====================================================

# Estructura esperada:
# TFM-Analisis-Inmobiliario/
# ├── data/raw
# ├── data/processed
# ├── notebooks
# └── reports/{figures,tables}
#
# La detección funciona tanto si VS Code ejecuta el notebook
# desde la raíz del proyecto como desde la carpeta notebooks/.
cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        ruta for ruta in [cwd, *cwd.parents]
        if (ruta / "data" / "raw").is_dir()
        and (ruta / "notebooks").is_dir()
        and (ruta / "reports").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Se esperaba encontrar las carpetas data/raw, notebooks y reports "
        "en el directorio actual o en alguno de sus directorios superiores."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Directorio de trabajo: {cwd}")
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Datos originales: {RAW_DATA_DIR}")
print(f"Datos procesados: {PROCESSED_DATA_DIR}")
print(f"Figuras: {FIGURES_DIR}")


### Interpretación

La configuración concentra las rutas utilizadas en todo el proyecto. Los notebooks posteriores conservan la misma estructura para garantizar la reproducibilidad del análisis.


## 3. Localización y carga de los archivos

Los archivos de las ciudades cuyo nombre termina en `2026.xlsx` se localizan de forma recursiva. Esta búsqueda funciona tanto si los Excel están directamente en `data/raw` como si cada ciudad aparece en una subcarpeta.

Las rutas de los archivos localizados se conservan para reutilizarlas en las comprobaciones posteriores. Durante la revisión estructural se leen únicamente los elementos necesarios para comprobar las columnas y contar los registros; los archivos completos se cargan posteriormente durante la integración.


In [ ]:
# =====================================================
# LOCALIZACIÓN DE LOS ARCHIVOS
# =====================================================

excel_files = sorted(
    p for p in RAW_DATA_DIR.rglob("* 2026.xlsx")
    if "INE" not in p.parts
)

print(f"Se han encontrado {len(excel_files)} archivos de ciudades.")
for file in excel_files:
    print(f"- {file.name}")

if not excel_files:
    raise FileNotFoundError(
        "No se han encontrado archivos de ciudades. "
        f"Revisa que los Excel estén dentro de: {RAW_DATA_DIR}"
    )

if len(excel_files) != 10:
    print(
        "\nAVISO: se esperaban 10 archivos de ciudades. "
        "Conviene revisar el contenido de data/raw antes de continuar."
    )


### Interpretación

En este TFM se trabajan **diez ciudades**. La verificación previa del número de archivos evita integrar por error otros Excel del proyecto, como fuentes auxiliares del INE.


## 4. Revisión de la estructura de los conjuntos de datos

Antes de integrar los archivos, se compara el número de registros y de variables de cada ciudad. También se registran las columnas disponibles en cada conjunto para comprobar su grado de homogeneidad.


In [ ]:
# =====================================================
# RESUMEN DE LOS CONJUNTOS DE DATOS
# =====================================================

resumen_archivos = []
columnas_por_archivo = {}

for file in excel_files:
    # Leer la cabecera para identificar las variables.
    cabecera = pd.read_excel(file, nrows=0)
    columnas_por_archivo[file.stem] = set(
        cabecera.columns
    )

    # Para contar registros se lee únicamente la primera
    # columna, evitando cargar centenares de variables.
    primera_columna = pd.read_excel(
        file,
        usecols=[0],
    )

    n_registros = len(primera_columna)
    n_variables = len(cabecera.columns)

    resumen_archivos.append({
        "Ciudad": file.stem.replace(" 2026", ""),
        "Número de registros": n_registros,
        "Número de variables": n_variables,
    })

tabla_resumen = (
    pd.DataFrame(resumen_archivos)
    .sort_values("Ciudad")
    .reset_index(drop=True)
)

fila_total = pd.DataFrame({
    "Ciudad": ["Total"],
    "Número de registros": [
        tabla_resumen["Número de registros"].sum()
    ],
    "Número de variables": [pd.NA],
})

tabla_resumen = pd.concat(
    [tabla_resumen, fila_total],
    ignore_index=True,
)

display(tabla_resumen)

# Figura 3-1: número de registros y variables por ciudad
tabla_figura = tabla_resumen[
    tabla_resumen["Ciudad"].ne("Total")
].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(
    tabla_figura["Ciudad"],
    tabla_figura["Número de registros"],
)
axes[0].set_title("Número de registros por ciudad")
axes[0].set_xlabel("Número de registros")
axes[0].set_ylabel("Ciudad")

axes[1].barh(
    tabla_figura["Ciudad"],
    tabla_figura["Número de variables"],
)
axes[1].set_title("Número de variables por ciudad")
axes[1].set_xlabel("Número de variables")
axes[1].set_ylabel("Ciudad")

plt.tight_layout()
fig.savefig(
    FIGURES_DIR / "figura_3_01_registros_variables_por_ciudad.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


### Interpretación

Los diez archivos reúnen **55.653 anuncios** antes del preprocesamiento. El número de columnas no coincide exactamente en todas las ciudades, de modo que resulta necesario revisar qué variables son comunes y cuáles aparecen solo en determinados archivos.


## 5. Variables comunes y conjunto total de variables

Se calcula:

1. la **intersección** de columnas, es decir, las variables presentes en las diez ciudades;
2. la **unión** de columnas, que representa todas las variables detectadas en al menos una ciudad.

In [ ]:
# =====================================================
# VARIABLES COMUNES Y TOTALES
# =====================================================

variables_comunes = set.intersection(*columnas_por_archivo.values())
variables_totales = set.union(*columnas_por_archivo.values())

print(f"Variables comunes a las 10 ciudades: {len(variables_comunes)}")
print(f"Variables diferentes detectadas: {len(variables_totales)}")

variables_comunes_df = pd.DataFrame(
    sorted(variables_comunes),
    columns=["Variable común"],
)

variables_totales_df = pd.DataFrame(
    sorted(variables_totales),
    columns=["Variable disponible"],
)

display(variables_comunes_df.head(20))
display(variables_totales_df.head(20))

### Interpretación

Se identifican **320 variables comunes** a las diez ciudades y **534 variables distintas** antes de añadir la variable `ciudad`. Esto confirma la existencia de un núcleo amplio de información comparable, aunque también hay columnas específicas de ciertos archivos.


## 6. Frecuencia de aparición de las variables

La frecuencia indica en cuántas ciudades aparece cada variable. Esta revisión permite distinguir variables ampliamente disponibles de columnas muy específicas.


In [ ]:
# =====================================================
# FRECUENCIA DE APARICIÓN DE LAS VARIABLES
# =====================================================

frecuencia_variables = {}

for columnas in columnas_por_archivo.values():
    for variable in columnas:
        frecuencia_variables[variable] = frecuencia_variables.get(variable, 0) + 1

frecuencia_variables_df = (
    pd.DataFrame(
        frecuencia_variables.items(),
        columns=["Variable", "Número de ciudades"],
    )
    .sort_values(
        ["Número de ciudades", "Variable"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

resumen_frecuencia = (
    frecuencia_variables_df["Número de ciudades"]
    .value_counts()
    .sort_index()
    .rename_axis("Número de ciudades")
    .reset_index(name="Número de variables")
)

display(frecuencia_variables_df.head(20))
display(resumen_frecuencia)

### Interpretación

La frecuencia de aparición muestra que muchas variables están disponibles en todas las ciudades, mientras que otras son específicas de uno o de pocos archivos. La decisión de conservar una variable no se basará solo en su frecuencia, sino también en su significado y en su utilidad para el objetivo del TFM.


## 7. Variables exclusivas por ciudad

Se consideran exclusivas las variables presentes en una única ciudad. Se muestran tanto el número de variables exclusivas como su detalle.

In [ ]:
# =====================================================
# VARIABLES EXCLUSIVAS POR CIUDAD
# =====================================================

detalle_exclusivas = []

for ciudad, columnas in columnas_por_archivo.items():
    for variable in sorted(columnas):
        if frecuencia_variables[variable] == 1:
            detalle_exclusivas.append({
                "Ciudad": ciudad.replace(" 2026", ""),
                "Variable": variable,
            })

detalle_exclusivas_df = pd.DataFrame(detalle_exclusivas)

if detalle_exclusivas_df.empty:
    print("No se han detectado variables exclusivas.")
else:
    resumen_exclusivas = (
        detalle_exclusivas_df.groupby("Ciudad")
        .size()
        .rename("Variables exclusivas")
        .sort_values(ascending=False)
        .reset_index()
    )

    display(resumen_exclusivas)
    display(detalle_exclusivas_df)

### Interpretación

Las variables exclusivas se concentran sobre todo en información auxiliar de algunos anuncios, en especial recursos multimedia. Por tanto, que una ciudad disponga de más columnas no implica necesariamente que ofrezca más características relevantes de las viviendas.

Esta revisión será útil en el preprocesamiento para eliminar información técnica o multimedia que no contribuya al análisis inmobiliario.


## 8. Integración de los conjuntos de datos

Los diez DataFrames se integran en uno solo. Antes de concatenarlos se añade la variable `ciudad`, que identifica el origen de cada anuncio.

In [ ]:
# =====================================================
# INTEGRACIÓN DE LOS CONJUNTOS DE DATOS
# =====================================================

df_viviendas = None

for file in excel_files:
    df_ciudad = pd.read_excel(file)
    df_ciudad["ciudad"] = (
        file.stem.replace(" 2026", "")
    )

    if df_viviendas is None:
        df_viviendas = df_ciudad
    else:
        df_viviendas = pd.concat(
            [df_viviendas, df_ciudad],
            ignore_index=True,
            sort=False,
        )

    # Se libera la referencia temporal antes de
    # cargar el siguiente archivo.
    del df_ciudad

print(f"Número de registros: {df_viviendas.shape[0]:,}")
print(f"Número de variables: {df_viviendas.shape[1]}")
display(df_viviendas.head())

### Interpretación

La integración genera un conjunto de datos de **55.653 registros y 535 variables**. Las 534 variables originales se complementan con `ciudad`, necesaria para realizar después comparaciones territoriales.


## 9. Verificación final de la integración

Se revisa la distribución de registros por ciudad y se comprueba que la suma coincide con el número total de observaciones.

In [ ]:
# =====================================================
# VERIFICACIÓN FINAL
# =====================================================

registros_por_ciudad = (
    df_viviendas["ciudad"]
    .value_counts()
    .sort_index()
    .rename_axis("Ciudad")
    .reset_index(name="Número de registros")
)

display(registros_por_ciudad)
print(f"Comprobación del total: {registros_por_ciudad['Número de registros'].sum():,}")

### Interpretación

La suma de los registros por ciudad coincide con el total integrado, por lo que no se observan pérdidas de información durante la concatenación de los diez archivos.


## 10. Resultados principales

- Se localizaron y revisaron los **10 conjuntos de datos urbanos**.
- Los archivos contienen conjuntamente **55.653 anuncios**.
- Se identificaron **534 variables originales diferentes**.
- **320 variables** están presentes en las diez ciudades.
- Tras añadir `ciudad`, el conjunto integrado contiene **535 variables**.
- Una parte importante de las diferencias estructurales entre ciudades corresponde a información auxiliar o multimedia.


## 11. Conclusiones de la fase

La fase de selección confirma que los diez conjuntos de datos presentan un núcleo suficientemente homogéneo para realizar un análisis comparativo entre ciudades.

La integración aporta una base única de **55.653 anuncios y 535 variables**. Aunque existen diferencias en el número de columnas entre ciudades, una parte importante corresponde a información auxiliar o multimedia y no a características esenciales del inmueble.

El siguiente paso consiste en depurar esta base integrada: revisar valores ausentes, eliminar variables sin utilidad analítica, comprobar duplicados y construir el **conjunto de datos procesado que servirá de entrada al análisis exploratorio y a la modelización**.


In [ ]:
# =====================================================
# EXPORTACIÓN DE TABLAS DE SELECCIÓN E INTEGRACIÓN
# =====================================================

TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

tabla_resumen.to_csv(
    TABLES_DIR / "01_resumen_archivos_por_ciudad.csv",
    index=False,
    encoding="utf-8-sig",
)

variables_comunes_df.to_csv(
    TABLES_DIR / "01_variables_comunes.csv",
    index=False,
    encoding="utf-8-sig",
)

variables_totales_df.to_csv(
    TABLES_DIR / "01_variables_totales.csv",
    index=False,
    encoding="utf-8-sig",
)

frecuencia_variables_df.to_csv(
    TABLES_DIR / "01_frecuencia_variables.csv",
    index=False,
    encoding="utf-8-sig",
)

resumen_frecuencia.to_csv(
    TABLES_DIR / "01_resumen_frecuencia_variables.csv",
    index=False,
    encoding="utf-8-sig",
)

if not detalle_exclusivas_df.empty:
    resumen_exclusivas.to_csv(
        TABLES_DIR / "01_resumen_variables_exclusivas_por_ciudad.csv",
        index=False,
        encoding="utf-8-sig",
    )
    detalle_exclusivas_df.to_csv(
        TABLES_DIR / "01_detalle_variables_exclusivas.csv",
        index=False,
        encoding="utf-8-sig",
    )

registros_por_ciudad.to_csv(
    TABLES_DIR / "01_registros_integrados_por_ciudad.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Tablas del Notebook 01 exportadas en:", TABLES_DIR)
